<a href="https://colab.research.google.com/github/yamazaki-riko/python_learning/blob/koshien_google-colab/db_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# セル1：SQLファイルをアップロード
from google.colab import files
uploaded = files.upload()
# 実行するとファイル選択ダイアログが出るので
# koshien_full.sql を選んでください

Saving koshien_full.sql to koshien_full.sql


In [7]:
# ---------------------------------------------------------------
# セル2: SQLite変換関数を定義する
# ---------------------------------------------------------------

import sqlite3
import re
import pandas as pd

def load_postgres_to_sqlite(sql_file="koshien_full.sql",
                             db_file="koshien.db"):
    """
    PostgreSQLダンプ → SQLiteに変換してDBファイルを作成する
    戻り値: sqlite3.Connection
    """
    with open(sql_file, "r", encoding="utf-8") as f:
        content = f.read()

    conn = sqlite3.connect(db_file)
    cur  = conn.cursor()

    # ===== STEP A: CREATE TABLE =====
    print("テーブルを作成中...")
    create_pattern = re.compile(
        r'CREATE TABLE\s+koshien\.(\w+)\s*\((.+?)\)\s*;',
        re.DOTALL
    )
    for m in create_pattern.finditer(content):
        table_name = m.group(1)
        cols_raw   = m.group(2)

        # PostgreSQL型 → SQLite型 に変換
        cols = cols_raw
        cols = re.sub(r'character varying\(\d+\)', 'TEXT',    cols)
        cols = re.sub(r'character\(\d+\)',          'TEXT',    cols)
        cols = re.sub(r'numeric\(\d+,\d+\)',        'REAL',    cols)
        cols = re.sub(r'\bsmallint\b',              'INTEGER', cols)
        cols = re.sub(r'\binteger\b',               'INTEGER', cols)
        cols = re.sub(r'\btext\b',                  'TEXT',    cols)
        cols = re.sub(r'NOT NULL',                  '',        cols)

        create_sql = (
            f"CREATE TABLE IF NOT EXISTS {table_name} (\n"
            f"    {cols.strip()}\n);"
        )
        try:
            cur.execute(create_sql)
            print(f"  ✅ {table_name}")
        except Exception as e:
            print(f"  ❌ {table_name}: {e}")

    conn.commit()

    # ===== STEP B: COPY文 → INSERT =====
    print("\nデータを挿入中...")
    copy_pattern = re.compile(
        r'COPY\s+koshien\.(\w+)\s*\(([^)]+)\)\s+FROM\s+stdin;\n(.*?)^\\\.',
        re.DOTALL | re.MULTILINE
    )
    for m in copy_pattern.finditer(content):
        table_name  = m.group(1)
        cols_str    = m.group(2).strip()
        data_block  = m.group(3)

        cols_list    = [c.strip() for c in cols_str.split(',')]
        placeholders = ','.join(['?' for _ in cols_list])
        insert_sql   = (
            f"INSERT INTO {table_name} ({cols_str}) "
            f"VALUES ({placeholders})"
        )

        rows_ok = 0
        for line in data_block.splitlines():
            line = line.rstrip('\n')
            if not line:
                continue
            values_raw = line.split('\t')
            if len(values_raw) != len(cols_list):
                continue
            values = [None if v == '\\N' else v for v in values_raw]
            try:
                cur.execute(insert_sql, values)
                rows_ok += 1
            except Exception:
                pass

        conn.commit()
        print(f"  ✅ {table_name}: {rows_ok}行")

    print("\n🎉 変換完了！")
    return conn


print("✅ 関数の定義が完了しました")

✅ 関数の定義が完了しました


In [6]:
# ---------------------------------------------------------------
# セル3: 変換を実行する
# ---------------------------------------------------------------

conn = load_postgres_to_sqlite("koshien_full.sql", "koshien.db")

テーブルを作成中...
  ✅ tmp_coaching_career
  ✅ tmp_koshien_appearances_summer
  ✅ tmp_media_rank
  ✅ tmp_team_batting_stats
  ✅ tmp_team_pitching_stats
  ✅ tmp_tournament_games

データを挿入中...
  ✅ tmp_coaching_career: 35行
  ✅ tmp_koshien_appearances_summer: 196行
  ✅ tmp_media_rank: 196行
  ✅ tmp_team_batting_stats: 296行
  ✅ tmp_team_pitching_stats: 296行
  ✅ tmp_tournament_games: 208行

🎉 変換完了！


In [9]:
# ---------------------------------------------------------------
# セル4: テーブルの中身を確認する
# ---------------------------------------------------------------

# テーブル一覧と行数
print("📋 テーブル一覧:")
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)
for t in tables["name"]:
    cnt = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {t}", conn)["cnt"][0]
    print(f"  {t}: {cnt}行")

print()

# 試合結果を確認
print("⚾ 試合結果（最初の5件）:")
df_games = pd.read_sql("""
    SELECT year, game_date, round_name,
           winner_school, winner_score,
           loser_score,  loser_school,
           remarks
    FROM tmp_tournament_games
    ORDER BY year, game_date
    LIMIT 5
""", conn)
display(df_games)

print("📰 メディアランキング（2025年、上位5校）:")
df_media = pd.read_sql("""
    SELECT school_name, media_avg_score, media_rank,
           nikkan_sports_rating, sponichi_rating,
           sports_hochi_rating
    FROM tmp_media_rank
    WHERE year = 2025
    ORDER BY media_rank
    LIMIT 5
""", conn)
display(df_media)

print("🏫 甲子園出場回数（2025年、出場回数トップ5）:")
df_app = pd.read_sql("""
    SELECT school_name, district, summer_appearances
    FROM tmp_koshien_appearances_summer
    WHERE year = 2025
    ORDER BY summer_appearances DESC
    LIMIT 5
""", conn)
display(df_app)

📋 テーブル一覧:
  tmp_coaching_career: 35行
  tmp_koshien_appearances_summer: 196行
  tmp_media_rank: 196行
  tmp_team_batting_stats: 296行
  tmp_team_pitching_stats: 296行
  tmp_tournament_games: 208行

⚾ 試合結果（最初の5件）:


,year,game_date,round_name,winner_school,winner_score,loser_score,loser_school,remarks
0,2022,8月10日,1回戦,仙台育英,10,0,鳥取商,None
1,2022,8月10日,1回戦,明秀日立,2,1,鹿児島実,None
2,2022,8月10日,1回戦,旭川大高,5,4,能代松陽,None
3,2022,8月10日,1回戦,大阪桐蔭,12,2,聖望学園,None
4,2022,8月11日,1回戦,高松商,14,4,佐久長聖,None


📰 メディアランキング（2025年、上位5校）:


,school_name,media_avg_score,media_rank,nikkan_sports_rating,sponichi_rating,sports_hochi_rating
0,健大高崎,4.2,1,S,A,A
1,関東一,4.0,2,A,A,A
2,東海大相模,4.0,2,A,A,A
3,横浜,4.0,2,A,A,A
4,京都国際,4.0,2,A,A,A


🏫 甲子園出場回数（2025年、出場回数トップ5）:


,school_name,district,summer_appearances
0,広陵,広島,35
1,星稜,石川,27
2,横浜,西東京,26
3,熊本工,熊本,24
4,北海,南北海道,23


In [10]:
# ---------------------------------------------------------------
# セル5: Google Drive に保存する（セッション切れ対策）
# ---------------------------------------------------------------

from google.colab import drive
import shutil, os

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/koshien"
os.makedirs(save_dir, exist_ok=True)

shutil.copy("koshien.db", f"{save_dir}/koshien.db")
print(f"✅ Drive に保存しました: {save_dir}/koshien.db")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━
次回以降はこのコードでDBを開けます:

  from google.colab import drive
  import sqlite3, pandas as pd

  drive.mount('/content/drive')
  conn = sqlite3.connect("/content/drive/MyDrive/koshien/koshien.db")

  df = pd.read_sql("SELECT * FROM tmp_tournament_games LIMIT 5", conn)
  display(df)
━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

Mounted at /content/drive
✅ Drive に保存しました: /content/drive/MyDrive/koshien/koshien.db

━━━━━━━━━━━━━━━━━━━━━━━━━━
次回以降はこのコードでDBを開けます:
 
  from google.colab import drive
  import sqlite3, pandas as pd
 
  drive.mount('/content/drive')
  conn = sqlite3.connect("/content/drive/MyDrive/koshien/koshien.db")
 
  df = pd.read_sql("SELECT * FROM tmp_tournament_games LIMIT 5", conn)
  display(df)
━━━━━━━━━━━━━━━━━━━━━━━━━━

